# HyperStreamDB: Full Pipeline Presentation

**Version:** 0.1.12 (Alpha)  
**Project:** HyperStreamDB - Serverless Index-Streaming Data Lake

This masterclass takes you from zero to a production-grade RAG pipeline, demonstrating hardware acceleration, hybrid search, and serverless table management.

## 1. Setup & Environment

We initialize our environment and ensure all vector search dependencies are loaded.

In [11]:
import hyperstreamdb as hdb
import pyarrow as pa
import pandas as pd
import numpy as np
import os, shutil, requests, time, warnings
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 150)

print("Environment ready.")

Environment ready.


## 2. Hardware Acceleration

HyperStreamDB can utilize Intel GPUs for massively parallel index construction. We check for available hardware here.

GPU support for Nvidia, Intel, Apple MPS, AMD(ROCm - untested)

In [12]:
import torch
device = "cpu"
if torch.cuda.is_available():
    device = "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")
ctx = hdb.Device(device)


Using device: cpu


## 3. Data Preparation (AG News)

We load the AG News dataset and generate semantic embeddings using a 384-dimensional SBERT model.

In [13]:
print("Loading dataset...")
dataset = load_dataset("ag_news", split="test[:500]")
df = pd.DataFrame(dataset)
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
df["category"] = df["label"].map(label_map)

print("Generating embeddings...")
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["text"].tolist())
df["embedding"] = [list(e) for e in embeddings]
print(f"Prepared {len(df)} records with {len(df['embedding'][0])}D vectors.")

df["id"] = range(len(df))


Loading dataset...
Generating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prepared 500 records with 384D vectors.


## 4. Table Creation (Serverless Iceberg)

HyperStreamDB tables are serverless and use standard Iceberg/Parquet formats with sidecar indices.

In [14]:
TABLE_URI = "file:///tmp/hdb_masterclass"
if os.path.exists("/tmp/hdb_masterclass"): shutil.rmtree("/tmp/hdb_masterclass")

schema = pa.schema([
    ('id', pa.int32()), 
    ('category', pa.string()),
    ('text', pa.string()),
    ('embedding', pa.list_(pa.float32(), 384))
])

table = hdb.Table.create(TABLE_URI, schema, device=ctx)
print(f"Table created at {TABLE_URI}")

Table created at file:///tmp/hdb_masterclass


## 5. Ingestion & Indexing

We write the data and building HNSW vector indices and Inverted scalar indices simultaneously.

In [15]:
table.add_index_columns(["embedding", "category", "text"])
table.write(df[["id", "category", "text", "embedding"]])
table.commit()
print("Data ingested and indexed successfully.")

Optimizing data layout: Shuffling rows by vector similarity (LanceDB-style)...
Written data to /tmp/hdb_masterclass/seg_65b26032-619b-44d3-b4c6-50061e29adee.parquet (500 rows)
Building indexes for batch of 500 rows
Indexing Int32 column: id
Indexing String column: text
Indexing String column: category
Indexing Vector column: embedding (type=FixedSizeList(Field { data_type: Float32, nullable: true }, 384))
  Found 4 unique values
Building HNSW-IVF index (blocking): 500 vectors, 384 dims, use_pq=false
  Found 500 unique values
String Inverted Index written to /tmp/hdb_masterclass/seg_65b26032-619b-44d3-b4c6-50061e29adee.category.inv.parquet
String Inverted Index written to /tmp/hdb_masterclass/seg_65b26032-619b-44d3-b4c6-50061e29adee.text.inv.parquet
Data ingested and indexed successfully.


DEBUG: to_manifest_entry: segment_id=seg_65b26032-619b-44d3-b4c6-50061e29adee, generated index_files=[IndexFile { file_path: "seg_65b26032-619b-44d3-b4c6-50061e29adee.id.inv.parquet", index_type: "inverted", column_name: Some("id"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_65b26032-619b-44d3-b4c6-50061e29adee.category.inv.parquet", index_type: "inverted", column_name: Some("category"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_65b26032-619b-44d3-b4c6-50061e29adee.text.inv.parquet", index_type: "inverted", column_name: Some("text"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_65b26032-619b-44d3-b4c6-50061e29adee.embedding.cluster_2.hnsw.graph", index_type: "vector", column_name: Some("embedding"), blob_type: None, offset: None, length: None }]
DEBUG: Background indexing task: segment=seg_65b26032-619b-44d3-b4c6-50061e29adee.parquet, found index_files=[IndexFile { file_path: "seg_65b26032-61

## 6. Hybrid Search Performance

We perform a semantic search combined with a scalar filter to find 'Space' news in the 'Sci/Tech' category.

In [16]:
query_vec = list(model.encode("Latest milestones in space exploration"))

results = (table.filter("category = 'Sci/Tech'")
                 .vector_search(query_vec, k=3)
                 .to_pandas())

print("Hybrid Search Results (Sci/Tech + Vector):")
display(results[["category", "text", "distance"]])

Hybrid Search Results (Sci/Tech + Vector):


,category,text,distance
0,Sci/Tech,"The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers ...",1.113737
1,Sci/Tech,Stunt pilots to snag a falling NASA craft NASA #39;s three-year effort to bring some genuine star dust back to Earth is set for a dramatic finale ...,1.254462
2,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space Agency -- ESAs Mars Express has relayed pictures from one of NASA's Mars rovers for t...,1.255060


## 7. SQL & pgvector Interface

HyperStreamDB supports standard SQL with pgvector-compatible operators for vector distance.

In [17]:
session = hdb.Session()
session.register("news", table)

query_vec_str = str([float(x) for x in query_vec]).replace(' ', '')
sql_df = session.sql(f"""
    SELECT category, text, 
           embedding <-> '{query_vec_str}'::vector AS dist
    FROM news
    WHERE category = 'Sci/Tech'
    ORDER BY dist ASC LIMIT 3
""")
display(sql_df.to_pandas())

,category,text,dist
0,Sci/Tech,"The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers ...",1.055337
1,Sci/Tech,Stunt pilots to snag a falling NASA craft NASA #39;s three-year effort to bring some genuine star dust back to Earth is set for a dramatic finale ...,1.120027
2,Sci/Tech,Mars Rovers Relay Images Through Mars Express European Space Agency -- ESAs Mars Express has relayed pictures from one of NASA's Mars rovers for t...,1.120295


## 8. Vector Aggregations

Calculate semantic centroids for entire categories directly in SQL.

In [18]:
agg_df = session.sql("""
    SELECT category, 
           vector_avg(embedding) AS centroid,
           COUNT(*) as count
    FROM news
    GROUP BY category
    order by count asc
""").to_pandas()
display(agg_df[["category", "count"]])

,category,count
0,Business,106
1,World,122
2,Sci/Tech,127
3,Sports,145


## 9. Table Maintenance

Keep the data lake healthy by compacting small files and vacuuming history.

In [19]:
print(f"Files before: {len(table.manifest().entries)}")
table.compact()
table.commit()
print(f"Files after: {len(table.manifest().entries)}")

table.vacuum(retention_versions=1)
print("Vacuum complete.")

Files before: 0
Written data to /tmp/hyperstream_compact_8a6dea1c-033f-4ef8-9b87-f2a7c6bde647/compacted_20260411020603_8a6dea1c-033f-4ef8-9b87-f2a7c6bde647.parquet (500 rows)
Files after: 0
Vacuum complete.


## 10. Advanced Partitioning

Scale to massive datasets using identity partitioning on the category field.

In [20]:
PART_URI = "file:///tmp/part_demo"
if os.path.exists("/tmp/part_demo"): shutil.rmtree("/tmp/part_demo")

partition_spec = {
    "fields": [{"name": "category", "transform": "identity", "source_id": 1, "field_id": 1000}]
}

part_table = hdb.Table.create_partitioned(PART_URI, schema, partition_spec)
part_table.write(df[:100])
part_table.commit()

print("Partitioned physical layout:")
for entry in part_table.manifest().entries:
    print(f" - {entry.file_path.split('/')[-2:]}")

DEBUG: Rust Table created with runtime: true
Optimizing data layout: Shuffling rows by vector similarity (LanceDB-style)...
Written data to /tmp/part_demo/category=Business/seg_3a6cb465-6022-4f05-853e-268d1d0c42cd.parquet (12 rows)
Written data to /tmp/part_demo/category=Sports/seg_e785e154-0779-4bda-99f9-0ed5993c386a.parquet (21 rows)
Written data to /tmp/part_demo/category=World/seg_1a415f09-fbc3-491e-95be-5678fca0ef5e.parquet (30 rows)
Written data to /tmp/part_demo/category=Sci/Tech/seg_fe4b287f-e3b5-4bca-b7ae-9bf2cb0c13a1.parquet (37 rows)
Building indexes for batch of 12 rows
Building indexes for batch of 30 rows
Building indexes for batch of 21 rows
Indexing Int32 column: id
Indexing String column: category
Indexing String column: text
  Found 1 unique values
Indexing Vector column: embedding (type=FixedSizeList(Field { data_type: Float32, nullable: true }, 384))
  Found 12 unique values
Indexing Int32 column: id
Indexing String column: text
Building indexes for batch of 37 rows

RuntimeError: Value Record([("status", Int(1)), ("snapshot_id", Union(1, Long(2))), ("sequence_number", Union(1, Long(2))), ("file_sequence_number", Union(1, Long(2))), ("data_file", Record([("content", Int(0)), ("file_path", String("category=World/seg_1a415f09-fbc3-491e-95be-5678fca0ef5e.parquet")), ("file_format", String("PARQUET")), ("partition", Record([("category", Union(1, String("World")))])), ("record_count", Long(30)), ("file_size_in_bytes", Long(68555)), ("column_sizes", Union(0, Null)), ("value_counts", Union(0, Null)), ("null_value_counts", Union(0, Null)), ("nan_value_counts", Union(0, Null)), ("lower_bounds", Union(0, Null)), ("upper_bounds", Union(0, Null)), ("equality_ids", Union(0, Null)), ("index_files", Union(0, Null))]))]) does not match schema Record(RecordSchema { name: Name { name: "manifest", namespace: None }, aliases: None, doc: None, fields: [RecordField { name: "status", doc: Some("0=EXISTING, 1=ADDED, 2=DELETED"), aliases: None, default: None, schema: Int, 

DEBUG: to_manifest_entry: segment_id=seg_3a6cb465-6022-4f05-853e-268d1d0c42cd, generated index_files=[IndexFile { file_path: "seg_3a6cb465-6022-4f05-853e-268d1d0c42cd.category.inv.parquet", index_type: "inverted", column_name: Some("category"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_3a6cb465-6022-4f05-853e-268d1d0c42cd.id.inv.parquet", index_type: "inverted", column_name: Some("id"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_3a6cb465-6022-4f05-853e-268d1d0c42cd.text.inv.parquet", index_type: "inverted", column_name: Some("text"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_3a6cb465-6022-4f05-853e-268d1d0c42cd.embedding.cluster_0.hnsw.graph", index_type: "vector", column_name: Some("embedding"), blob_type: None, offset: None, length: None }]
DEBUG: Background indexing task: segment=category=Business/seg_3a6cb465-6022-4f05-853e-268d1d0c42cd.parquet, found index_files=[IndexFile { file_path

## 11. End-to-End RAG Pipeline

Using SQuAD validation data to build a knowledge-backed Q&A system.

In [21]:
print("Loading SQuAD contexts...")
squad = load_dataset("squad", split="validation[0:100]")
squad_df = pd.DataFrame(squad).drop_duplicates(subset=["context"])

squad_emb = model.encode(squad_df["context"].tolist(), show_progress_bar=True)
squad_df["embedding"] = [list(e) for e in squad_emb]

rag_table = hdb.Table("rag_db_presentation")
rag_table.add_index_columns(["embedding", "context"])
rag_table.write(squad_df[["context", "title", "embedding"]])
rag_table.commit()
print("RAG knowledge base initialized.")

Loading SQuAD contexts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Written data to /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_1b943282-93e5-445d-8a80-1ab2e9d64294.parquet (4 rows)
Building indexes for batch of 4 rows
Indexing String column: context
Indexing Vector column: embedding (type=List(Field { data_type: Float32, nullable: true }))
Indexing String column: title
Indexing Int64 column: __index_level_0__
  Found 1 unique values
  Found 4 unique values
Building HNSW-IVF index (blocking): 4 vectors, 384 dims, use_pq=false
  [K-Means] Converged early at iteration 2
String Inverted Index written to /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_1b943282-93e5-445d-8a80-1ab2e9d64294.context.inv.parquet
String Inverted Index written to /home/ralbright/projects/hyperstreamdb/demo/rag_db_presentation/seg_1b943282-93e5-445d-8a80-1ab2e9d64294.title.inv.parquet
RAG knowledge base initialized.


DEBUG: to_manifest_entry: segment_id=seg_1b943282-93e5-445d-8a80-1ab2e9d64294, generated index_files=[IndexFile { file_path: "seg_1b943282-93e5-445d-8a80-1ab2e9d64294.__index_level_0__.inv.parquet", index_type: "inverted", column_name: Some("__index_level_0__"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_1b943282-93e5-445d-8a80-1ab2e9d64294.context.inv.parquet", index_type: "inverted", column_name: Some("context"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_1b943282-93e5-445d-8a80-1ab2e9d64294.title.inv.parquet", index_type: "inverted", column_name: Some("title"), blob_type: None, offset: None, length: None }, IndexFile { file_path: "seg_1b943282-93e5-445d-8a80-1ab2e9d64294.embedding.cluster_0.hnsw.graph", index_type: "vector", column_name: Some("embedding"), blob_type: None, offset: None, length: None }]
DEBUG: Background indexing task: segment=seg_1b943282-93e5-445d-8a80-1ab2e9d64294.parquet, found index_files=[IndexFile

## 12. Inference with Retrieval

Connect retrieval to the LLM (requires GROQ_API_KEY).

In [22]:
def ask_hdb(question):
    # Convert question to an embedding (using your local model)
    q_emb = model.encode(question)
    
    # Perform a vector search on your table
    # This is where your table data gets pulled in!
    ctx_df = rag_table.to_pandas(vector_filter={
        "column": "embedding", 
        "query": q_emb, 
        "k": 3  # Get top 3 matching contexts
    })
    
    # Combine retrieved contexts into one string
    contexts = ctx_df["context"].tolist()
    prompt = f"Answer the question based ONLY on this context:\n\n" + "\n".join(contexts) + f"\n\nQuestion: {question}"
    
    # Use a supported Groq model
    model_name = "llama-3.3-70b-versatile"
    
    api_key = os.environ.get("GROQ_TOKEN")
    r = requests.post("https://api.groq.com/openai/v1/chat/completions", 
                      headers={"Authorization": f"Bearer {api_key}"},
                      json={"model": model_name, "messages": [{"role": "user", "content": prompt}]})
    
    res = r.json()
    if "choices" not in res:
        # This will show you the exact error from Groq (e.g., "model_not_found")
        print(f"Groq API Error: {res}")
        return "Error from API"
        
    return res["choices"][0]["message"]["content"]

In [23]:

question = "Where were the first modern Olympic games?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

Q: Where were the first modern Olympic games?
A: This context does not provide information about the first modern Olympic games. It is about Super Bowl 50.


In [24]:
question = "What city was superbowl 50 played at?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

Q: What city was superbowl 50 played at?
A: The game was played at Santa Clara, California, but more specifically, it was held at Levi's Stadium in the San Francisco Bay Area.
